# FOLIO Shared Config + Login

This notebook holds the FOLIO connection settings (Okapi URL, tenant, username) and the login logic, so other notebooks don't need to repeat this action. 

**To use if from another notebook,** both notebooks must be in the same folder. Then, include the following  in the first line of your notebook:

%run folio_auth.ipynb

In [ ]:
import requests
import getpass
import os

# clear out any previous sesion data
session=''
token=''

## Configuration

Edit the values below for your FOLIO tenant. Any notebook that first calls this notebook will inherit values as global variables, as well as the authentication token you need to access the FOLIO API.

In [ ]:
# --- EDIT THESE THREE VALUES ---
OKAPI_URL   = "https://api-your-institution.folio.ebsco.com"    # Your FOLIO API gateway URL (no trailing slash)
TENANT      = "fs0000000"                                       # Your FOLIO tenant ID
USERNAME    = "your-username"                                   # Your FOLIO username
PASSWORD    = os.environ.get("FOLIO_PASSWORD")
# Storing your password as an environment variable is more secure than listing it here in the notebook.
# If FOLIO_PASSWORD is not set, PASSWORD will be None and you will be prompted securely (via getpass) below.
#
# How to set FOLIO_PASSWORD as an environment variable before launching Jupyter:
#
#   macOS / Linux (bash or zsh):
#     export FOLIO_PASSWORD='your-password-here'
#     (add this line to ~/.bashrc or ~/.zshrc to make it persist across sessions)
#
#   Windows (PowerShell):
#     $env:FOLIO_PASSWORD = 'your-password-here'        # current session only
#     setx FOLIO_PASSWORD "your-password-here"          # persists, but requires opening a NEW terminal
#
#   Windows (Command Prompt):
#     set FOLIO_PASSWORD=your-password-here             # current session only
#
# If you skip all of this, the notebook will just ask for your password interactively -- that's fine too.

In [ ]:
class FolioAuthError(Exception):
    """Raised when FOLIO login fails or the expected token cookie is missing."""
    pass


def folio_login(okapi_url, tenant, username, password):
    """
    Log in to FOLIO via the cookie-based /authn/login-with-expiry endpoint.

    Returns (session, token):
        session : requests.Session with the auth cookie attached and
                  X-Okapi-Tenant / X-Okapi-Token set as default headers
        token   : the raw access token string

    Raises FolioAuthError if login fails or no folioAccessToken cookie
    is found in the response.
    """
    if password is None:
        password = getpass.getpass(f"FOLIO password for {username}: ")

    session = requests.Session()

    login_url = f"{okapi_url}/authn/login-with-expiry"
    headers = {
        "X-Okapi-Tenant": tenant,
        "Content-Type": "application/json",
    }
    payload = {
        "username": username,
        "password": password,
    }

    response = session.post(login_url, headers=headers, json=payload)

    if response.status_code not in (200, 201):
        raise FolioAuthError(
            f"Login failed with status {response.status_code}: {response.text}"
        )

    token = session.cookies.get("folioAccessToken")

    if not token:
        raise FolioAuthError(
            "Login returned a success status but no 'folioAccessToken' cookie "
            f"was found. Cookies received: {session.cookies.get_dict()}. "
            "Your tenant may use a different cookie name -- check with your "
            "FOLIO admin or the response above."
        )

    session.headers.update({
        "X-Okapi-Tenant": tenant,
        "X-Okapi-Token": token,
    })

    return session, token

In [ ]:
class FolioAuthError(Exception):
    """Raised when FOLIO login fails or the expected token cookie is missing."""
    pass


def folio_login(okapi_url, tenant, username, password=None):
    """
    Log in to FOLIO via the cookie-based /authn/login-with-expiry endpoint.

    Returns (session, token):
        session : requests.Session with the auth cookie attached and
                  X-Okapi-Tenant / X-Okapi-Token set as default headers
        token   : the raw access token string

    Raises FolioAuthError if login fails or no folioAccessToken cookie
    is found in the response.
    """
    if password is None:
        password = getpass.getpass(f"FOLIO password for {username}: ")

    session = requests.Session()

    login_url = f"{okapi_url}/authn/login-with-expiry"
    headers = {
        "X-Okapi-Tenant": tenant,
        "Content-Type": "application/json",
    }
    payload = {
        "username": username,
        "password": password,
    }

    response = session.post(login_url, headers=headers, json=payload)

    if response.status_code not in (200, 201):
        raise FolioAuthError(
            f"Login failed with status {response.status_code}: {response.text}"
        )

    token = session.cookies.get("folioAccessToken")

    if not token:
        raise FolioAuthError(
            "Login returned a success status but no 'folioAccessToken' cookie "
            f"was found. Cookies received: {session.cookies.get_dict()}. "
            "Your tenant may use a different cookie name -- check with your "
            "FOLIO admin or the response above."
        )

    session.headers.update({
        "X-Okapi-Tenant": tenant,
        "X-Okapi-Token": token,
    })

    return session, token

## Log In

The function below is run automatically whenever this notebook is called from elsewhere. 

In [ ]:
try:
    session, token = folio_login(
        okapi_url=OKAPI_URL,
        tenant=TENANT,
        username=USERNAME,
        password=PASSWORD
    )
    print("Login succeeded. Token retrieved.")

except FolioAuthError as e:
    session, token = None, None
    print(f"Login failed: {e}")